# Taller: POO, Interfaces Gráficas y Bases de Datos (Supabase)

En esta lección vamos a crear una aplicación completa utilizando **Python**. Aprenderemos a conectar nuestro programa con **Supabase**, una alternativa moderna y de código abierto a Firebase (basada en PostgreSQL), y a crear una interfaz visual con **Tkinter**.

## 1. Definiendo nuestra Entidad (El Backend y POO)
Al igual que en otros sistemas, usamos Programación Orientada a Objetos para crear un molde (Clase) que representará a la información a guardar en la base de datos.

In [ ]:
class Usuario:
    def __init__(self, nombre, telefono, edad, comida_favorita):
        self.nombre = nombre
        self.telefono = telefono
        self.edad = edad
        self.comida_favorita = comida_favorita
        
    # Necesitamos convertir nuestro Objeto a un Diccionario
    # porque las APIs en internet (como la de Supabase) se comunican mediante JSON/diccionarios.
    def to_dict(self):
        return {
            "nombre": self.nombre,
            "telefono": self.telefono,
            "edad": int(self.edad),              # Aseguramos que sea número entero
            "comida_favorita": self.comida_favorita
        }

## 2. La Lógica de Conexión a Supabase

Supabase se basa en **PostgreSQL**, una potente base de datos relacional. Antes de ejecutar esto, debes instalar la librería oficial de Supabase para Python ejecutando en la terminal: `pip install supabase`

**IMPORTANTE:**
1. Entra a supabase.com, crea un proyecto.
2. Ve a 'Table Editor' y crea una tabla llamada `usuarios`.
3. Añade las columnas `nombre` (text), `telefono` (text), `edad` (int4), `comida_favorita` (text).
4. Desactiva RLS ('Row Level Security') temporalmente para pruebas sencillas, o crea policies públicas.
5. Copia tu URL y API Key en las variables de abajo.

In [ ]:
from supabase import create_client, Client
# pip install supabase

# Reemplaza estos valores con la URL y Key (anon public) de tu proyecto en Supabase
SUPABASE_URL = "https://TU-PROYECTO.supabase.co"
SUPABASE_KEY = "TU_CLAVE_ANON_AQUI"

# Creamos el cliente global de Supabase
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

class GestorUsuarios:
    def __init__(self):
        self.lista_usuarios = []
        self.usuarios_nuevos = []  # Llevaremos rastro de los que insertemos en esta sesión
        
    def agregar_localmente(self, usuario):
        self.lista_usuarios.append(usuario)
        self.usuarios_nuevos.append(usuario)

    def cargar_desde_supabase(self):
        print("Conectando a Supabase para descargar datos...")
        try:
            # Un query '.select("*")' trae todos los registros de la tabla 'usuarios'
            respuesta = supabase.table("usuarios").select("*").execute()
            
            # Los datos están alojados en la propiedad 'data' de la respuesta
            datos = respuesta.data
            self.lista_usuarios = []
            
            for val in datos:
                # Reconstruimos objetos Usuario desde los datos que nos envía Supabase
                obj_usuario = Usuario(val.get('nombre', ''), val.get('telefono', ''), 
                                      val.get('edad', 0), val.get('comida_favorita', ''))
                self.lista_usuarios.append(obj_usuario)
                
            print(f"¡Éxito! Se descargaron {len(self.lista_usuarios)} usuarios de la base de datos al iniciar el programa.")
        except Exception as e:
            print("Error al intentar cargar datos de Supabase:", e)

    def guardar_en_supabase(self):
        """Ésta función se llamará al cerrar nuestra ventana visual"""
        if not self.usuarios_nuevos:
            print("No hay usuarios nuevos para enviar a Supabase.")
            return
            
        print(f"Subiendo {len(self.usuarios_nuevos)} nuevos usuarios a Supabase...")
        
        # Transformamos nuestra sub-lista de objetos nuevos a diccionarios
        lista_a_insertar = [u.to_dict() for u in self.usuarios_nuevos]
            
        try:
            # Un query '.insert()' manda a agregar varios registros en una sola operación (bulk insert)
            respuesta = supabase.table("usuarios").insert(lista_a_insertar).execute()
            print("¡Nuevos datos respaldados exitosamente a la base de datos remota!")
        except Exception as e:
            print("No se pudo guardar la información a Supabase:", e)

## 3. La Interfaz Gráfica (El Frontend con Tkinter)
Conectaremos la lógica de Supabase a los botones de Tkinter. Aprovecharemos nuevamente la capacidad de capturar cuando se cierra la ventana (`WM_DELETE_WINDOW`) para guardar exclusivamente los **nuevos registros** a nuestra tabla relacional en la nube.

In [ ]:
import tkinter as tk
from tkinter import messagebox

class InterfazAplicacion:
    def __init__(self, ventana, gestor):
        self.ventana = ventana
        self.gestor = gestor
        self.ventana.title("Gestión de Perfiles - Supabase")
        
        # ----- PASO 1. AL INICIAR: CARGAR DESDE SUPABASE ----- 
        self.gestor.cargar_desde_supabase()
        
        # ----- PASO 2. AL CERRAR: GUARDAR ----- 
        self.ventana.protocol("WM_DELETE_WINDOW", self.evento_cerrar_ventana)
        
        # ----- DISEÑO DE LA APP (GRID SYSTEM EN TKINTER) -----
        tk.Label(ventana, text="Nombre:").grid(row=0, column=0, padx=10, pady=5)
        self.caja_nombre = tk.Entry(ventana)
        self.caja_nombre.grid(row=0, column=1, padx=10, pady=5)
        
        tk.Label(ventana, text="Teléfono:").grid(row=1, column=0, padx=10, pady=5)
        self.caja_tel = tk.Entry(ventana)
        self.caja_tel.grid(row=1, column=1, padx=10, pady=5)
        
        tk.Label(ventana, text="Edad:").grid(row=2, column=0, padx=10, pady=5)
        self.caja_edad = tk.Entry(ventana)
        self.caja_edad.grid(row=2, column=1, padx=10, pady=5)
        
        tk.Label(ventana, text="Comida Favorita:").grid(row=3, column=0, padx=10, pady=5)
        self.caja_comida = tk.Entry(ventana)
        self.caja_comida.grid(row=3, column=1, padx=10, pady=5)
        
        # Botones
        btn_agregar = tk.Button(ventana, text="Guardar Usuario (En Memoria Local)", command=self.agregar)
        btn_agregar.grid(row=4, column=0, columnspan=2, pady=10)
        
        btn_ver = tk.Button(ventana, text="Ver Lista (Ver Log en Consola)", command=self.mostrar)
        btn_ver.grid(row=5, column=0, columnspan=2, pady=5)

    def agregar(self):
        nom = self.caja_nombre.get()
        tel = self.caja_tel.get()
        ed = self.caja_edad.get()
        com = self.caja_comida.get()
        
        if nom and tel and ed and com:
            nuevo = Usuario(nom, tel, ed, com)
            self.gestor.agregar_localmente(nuevo)
            messagebox.showinfo("Aviso", "El usuario se agregó a la memoria con éxito.")
            
            self.caja_nombre.delete(0, tk.END)
            self.caja_tel.delete(0, tk.END)
            self.caja_edad.delete(0, tk.END)
            self.caja_comida.delete(0, tk.END)
        else:
            messagebox.showwarning("Error", "Debe llenar todos los campos antes de guardar.")
            
    def mostrar(self):
        print("\n--- Usuarios Actuales en Memoria ---")
        for u in self.gestor.lista_usuarios:
            print(f"{u.nombre} - Tel: {u.telefono} - Edad: {u.edad} - Comida: {u.comida_favorita}")
        print("------------------------------------\n")

    def evento_cerrar_ventana(self):
        if messagebox.askokcancel("Salir de la App", "¿Deseas cerrar y guardar toda la info recién agregada hacia Supabase?"):
            self.gestor.guardar_en_supabase()
            self.ventana.destroy()

## 4. Juntando Todo (Main)
Al igual que con Firebase, ejecutar esta última celda lanzará nuestra aplicación enlazada con la base de datos de PostgreSQL (Supabase).

In [ ]:
if __name__ == '__main__':
    # 1. Crear el cerebro de datos (Backend/Modelo)
    gestor_principal = GestorUsuarios()
    
    # 2. Crear el Contenedor Visual (Ventana Raíz GUI)
    ventana_raiz = tk.Tk()
    
    # 3. Lanzar la aplicación (pasándole la ventana para que se dibuje y el backend)
    app = InterfazAplicacion(ventana_raiz, gestor_principal)
    
    # 4. Iniciar el Loop incesante, deteniendo la ejecución hasta cerrar la ventana
    print("\n=> Aplicación Gráfica Activa. Cierra la ventana pop-up (con la X) para subir los nuevos registros (Insert) a la nube.\n")
    ventana_raiz.mainloop()